In [6]:
import pandas as pd

df = pd.read_parquet('../data/polygon/2026-02-23.parquet')

#-rw-r--r--@ 1 vobornij  staff   4,0M 10 bře 00:15 2025-12-04_79885902.parquet
#-rw-r--r--@ 1 vobornij  staff    14M 10 bře 00:53 2025-12-05_79885902.parquet
#-rw-r--r--@ 1 vobornij  staff    17M 10 bře 01:07 2025-12-06_79885902.parquet

print(f"length: {len(df)}")

# print head and tail, but convert timestamps to human-readable format

df_preview = pd.concat([df.head(2), df.tail(3)])
df_preview['block_timestamp'] = pd.to_datetime(df_preview['block_timestamp'], unit='s')
df_preview

length: 7006525


,block_number,tx_hash,log_index,block_timestamp,order_hash,maker,taker,maker_asset_id,taker_asset_id,maker_amount_filled,taker_amount_filled,fee
0,83342386,0xb807191fb959d399d30f11d9c431ad9483af95262ce8...,568,2026-02-23 00:00:00,0x07311ee9d2a9d8ce57c0a35adbc71868315d512480b6...,0x67ac9e1ad7d7e74ef0215d14fc8edb538e4fedf1,0x44e5a7a8205d1d5bbfa75616d4d6c4a043023c33,0,7991333529053676434468371960590677240171103216...,1454400,9090000,0
1,83342386,0xb807191fb959d399d30f11d9c431ad9483af95262ce8...,570,2026-02-23 00:00:00,0x03b718e7c6939fcd655d2d8992d8095261cdba4c2860...,0x44e5a7a8205d1d5bbfa75616d4d6c4a043023c33,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,7991333529053676434468371960590677240171103216...,0,9090000,1454400,0
7006522,83385585,0x7d9f9a949a69e887d6533415eaa51d378a55de7b6b80...,1702,2026-02-23 23:59:58,0x8e3e1bdbaee2cee6386501326af64697a03f6fd53ac8...,0x43e0cf2c6d6d71d9d6167332e07a9db7b534132a,0xa38a455bbdd4b68486548b7e19da99903f4f821d,0,4824842374131574622149087194048362492721063785...,2650000,5000000,0
7006523,83385585,0x7d9f9a949a69e887d6533415eaa51d378a55de7b6b80...,1710,2026-02-23 23:59:58,0x2ce5ab36946a88d7d393532d082743984111b4c9c455...,0xd37ab1a5eeb25696c568e20e94af72b8d3422358,0xa38a455bbdd4b68486548b7e19da99903f4f821d,0,4824842374131574622149087194048362492721063785...,1217872,2297871,0
7006524,83385585,0x7d9f9a949a69e887d6533415eaa51d378a55de7b6b80...,1712,2026-02-23 23:59:58,0x875d647702842eb4b168bedc9af7873863756710b494...,0xa38a455bbdd4b68486548b7e19da99903f4f821d,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,0,1914232705399838947008670500389070762547210572...,3429999,7297871,0


In [7]:
df[df['tx_hash'] == '0x21a446692c035c6f5a433c6ef1d2137fbb4d187c96f47a1b4dde82b8d8520b1e']

,block_number,tx_hash,log_index,block_timestamp,order_hash,maker,taker,maker_asset_id,taker_asset_id,maker_amount_filled,taker_amount_filled,fee
5235329,83376403,0x21a446692c035c6f5a433c6ef1d2137fbb4d187c96f4...,468,1771872834,0x68d14558f906346b464536a31de33d9b72e07b70559c...,0x74be214e7f0fa56c26425c438e366ed70fe0d9db,0x85d355ef32d62b09d2362184f299a7fc662ee1a4,0,5609504469749531682268396321654907199774022159...,10000,5000000,0
5235330,83376403,0x21a446692c035c6f5a433c6ef1d2137fbb4d187c96f4...,470,1771872834,0x45c9aefeffe877e5235c3ada669de2ce07c8acb70dc6...,0x85d355ef32d62b09d2362184f299a7fc662ee1a4,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,0,5419374323508319398707222459109139470874642588...,4990000,5000000,0


In [ ]:
# print numbers of trades in time, by 1 minute intervals
df['block_timestamp'] = pd.to_datetime(df['block_timestamp'], unit='s')
df.set_index('block_timestamp', inplace=True)
trades_per_minute = df.resample('1T').size()
trades_per_minute.plot()

In [ ]:
print(len(df['block_number'].unique()))

print(df['block_number'].max() - df['block_number'].min())
print(df['block_number'].min(), df['block_number'].max())


uniq = df['block_number'].unique()

len(uniq)

In [ ]:
import os
import pandas as pd

base_dir = "../data/polygon"

for file in sorted(os.listdir(base_dir)):
    if not file.endswith(".parquet"):
        continue

    path = os.path.join(base_dir, file)
    df = pd.read_parquet(path)

    if df.empty or "block_number" not in df.columns or "block_timestamp" not in df.columns:
        continue

    # important: reset index so positional neighbors are correct
    df_sorted = df.sort_values("block_number").reset_index(drop=True)
    df_sorted["block_number_diff"] = df_sorted["block_number"].diff()

    # ignore first NaN diff
    if df_sorted["block_number_diff"].iloc[1:].empty:
        continue

    max_pos = df_sorted["block_number_diff"].iloc[1:].idxmax()
    gap_size = int(df_sorted.at[max_pos, "block_number_diff"])

    # print only real gaps (>1 block)
    if gap_size <= 1:
        continue

    last_trade_before_gap = df_sorted.iloc[max_pos - 1]
    first_trade_after_gap = df_sorted.iloc[max_pos]

    before_ts = pd.to_datetime(last_trade_before_gap["block_timestamp"], unit="s")
    after_ts = pd.to_datetime(first_trade_after_gap["block_timestamp"], unit="s")

    print(
        f"Max gap in {file}: size {gap_size} "
        f"timestamps {before_ts} -> {after_ts}, "
        f"blocks {int(last_trade_before_gap['block_number'])} -> {int(first_trade_after_gap['block_number'])}"
    )

In [ ]:
# get numbers of trades per day and plot it

count_df = pd.DataFrame()
for file in os.listdir('../data/polygon')[-30:]:
    count_df = pd.concat([count_df, pd.DataFrame({'file': [file], 'count': [len(pd.read_parquet(os.path.join('../data/polygon', file)))]})])


In [ ]:
count_df.head(10)

In [ ]:
# print count, first and last timestamp
print("Count: " + str(len(df)))
print("First timestamp: " + str(pd.to_datetime(df['block_timestamp'].iloc[0], unit='s')))
print("Last timestamp: " + str(pd.to_datetime(df['block_timestamp'].iloc[-1], unit='s')))

In [65]:
files = [
    '/Users/vobornij/projects/polymarket/data/markets/2026-01.parquet',
    '/Users/vobornij/projects/polymarket/data/markets/2026-02.parquet',
    '/Users/vobornij/projects/polymarket/data/markets/2026-03.parquet'
]
mf = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
len(mf)

303103

In [66]:
# transform the market_json column that contains json text into separate columns
# we need to parse it and normalize
import json
from pandas import json_normalize
market_json_df = json_normalize(mf['market_json'].apply(json.loads))


In [68]:
# options set to display all rows
pd.set_option('display.max_rows', None)
market_json_df[['end_date_iso', 'market_slug', 'tags', 'question', 'tokens']].head(3)

,end_date_iso,market_slug,tags,question,tokens
0,2026-01-01T00:00:00Z,will-rasmr-win-the-synthetix-trading-competition-933,[Crypto],Will Rasmr win the Synthetix trading competition?,"[{'token_id': '67487832607016995738972990163663499867372390570812445168484453192396152205498', 'outcome': 'Yes', 'price': 0, 'winner': False}, {'token_id': '993932579711828259691151649663864465874..."
1,2026-01-01T00:00:00Z,btc-updown-15m-1767276000,"[Crypto, Bitcoin, Crypto Prices, Recurring, Up or Down, Hide From New, 15M]","Bitcoin Up or Down - January 1, 9:00AM-9:15AM ET","[{'token_id': '113678571358517111781856957181244877716762838026765852171442489235746182728379', 'outcome': 'Up', 'price': 1, 'winner': True}, {'token_id': '1107623073205307567090160469897810948102..."
2,2026-01-01T00:00:00Z,btc-updown-4h-1767258000,"[Crypto, Bitcoin, Crypto Prices, Recurring, Up or Down, Hide From New, 4H]","Bitcoin Up or Down - January 1, 4:00AM-8:00AM ET","[{'token_id': '35890450443256401466508231202457066327655991855628874696199993511781769449163', 'outcome': 'Up', 'price': 1, 'winner': True}, {'token_id': '11347810325984539627987368608969220093224..."


In [69]:
token_lookup = (
    market_json_df[["condition_id", "market_slug", "tags", "tokens", "end_date_iso"]]
    .explode("tokens", ignore_index=True)
    .assign(token_id=lambda d: d["tokens"].str["token_id"])
    .drop(columns=["tokens"])
    # .dropna(subset=["token_id"])
    # .drop_duplicates(subset=["token_id"])
    .set_index("token_id")[["condition_id", "market_slug", "tags", "end_date_iso"]]
)

token_lookup_dict = dict(
    zip(
        token_lookup.index,
        zip(token_lookup["condition_id"], token_lookup["market_slug"], token_lookup["tags"], token_lookup['end_date_iso'])
    )
)

len(token_lookup)

606206

In [75]:
type(token_lookup['tags'])

pandas.core.series.Series

In [76]:
# filtered_df = market_json_df[market_json_df['tags'].apply(lambda x: 'Crypto' not in x)]

len(token_lookup[token_lookup['tags'].apply(lambda x: isinstance(x, (list, tuple, set)) and ("Crypto" in x))])

153370

In [20]:
files = [
    '/Users/vobornij/projects/polymarket/data/polygon/2026-03-02.parquet',
]

trades_raw = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(len(trades_raw))

9107416


In [88]:
trades_raw['block_timestamp'] = pd.to_datetime(trades_raw['block_timestamp'], unit='s')
print(len(trades_raw))
trades_raw.sort_values('block_timestamp', inplace=True)
pd.concat([trades_raw.head(10), trades_raw.tail(3)])

9502654


,block_number,tx_hash,log_index,block_timestamp,order_hash,maker,taker,maker_asset_id,taker_asset_id,maker_amount_filled,taker_amount_filled,fee
0,83558385,0x80edc41388e388c84328fcd5ece2d1b069f6cc85e43d67fde2e4d848a79e9f24,328,2026-02-28 00:00:01,0xe2b0a7edd5c6c259d29b673ea229da5634d191a40913c298b21c504612a37cc6,0xab15040749c9a4ead610d6f0ba8f8da669137eac,0x076ca0207aaa48365da2eb6a917edcbdee5e4e7b,0,16581159092118702962412590502551978746865251171601054389828093232030979416235,9580040,258920000,0
1,83558385,0x80edc41388e388c84328fcd5ece2d1b069f6cc85e43d67fde2e4d848a79e9f24,330,2026-02-28 00:00:01,0x1ab32e2e558ac541e888533f92cf16f3e2fa99aa152404209b3b39058d39ccfc,0x076ca0207aaa48365da2eb6a917edcbdee5e4e7b,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,16581159092118702962412590502551978746865251171601054389828093232030979416235,0,258920000,9580040,0
2,83558385,0x47950bf4a4dc83fb7b82cf76c91aa68e275e92ce5b7e29c412710aa119a71f1d,344,2026-02-28 00:00:01,0x7a04a9bc97e7a5b75d12c41cd2b443eb914bdc03b84067559d28b4e867cd5f60,0x111f73e91f85b6fe4de1ddec3de2fe32122e355b,0xf6963d4cdbb6f26d753bda303e9513132afb1b7d,0,8629673814310781337262814398651414822607314895449171908987170554118708719632,7255600,10670000,502117
3,83558385,0x47950bf4a4dc83fb7b82cf76c91aa68e275e92ce5b7e29c412710aa119a71f1d,354,2026-02-28 00:00:01,0xcd829b6b817e9301138bc4738c1c7f7378f1f1e1ebbfe3d14f0fe3ea75e49bd7,0x36a708485c603a477804ec74122475647629f03b,0xf6963d4cdbb6f26d753bda303e9513132afb1b7d,0,8629673814310781337262814398651414822607314895449171908987170554118708719632,6400000,10000000,562500
4,83558385,0x47950bf4a4dc83fb7b82cf76c91aa68e275e92ce5b7e29c412710aa119a71f1d,358,2026-02-28 00:00:01,0x04aeb6a416faeb43d09276402977037869d9e69081f770bc24b53c6a60d61630,0xf6963d4cdbb6f26d753bda303e9513132afb1b7d,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,0,58910049878019002824701532570780342958758796534587879434389379308514509314738,7014400,20670000,2067000
5,83558385,0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0,378,2026-02-28 00:00:01,0xefe3172a3f9f9b6edf70c7e4bae007217be021e7ab97f79fc449c7496e95e867,0xefde78f38169634477f6ff46572128b6d21a6505,0x45730f93e26359aefad7fb6ee4851958d828c16b,0,51326343192525988136143382979855785783476474407776534721033463010659950298104,2883000,9610000,961000
6,83558385,0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0,388,2026-02-28 00:00:01,0x842b8ed0d2493b405f1eba864529c49a936304be9bb9e8e36fb986e523f89f26,0xd37ab1a5eeb25696c568e20e94af72b8d3422358,0x45730f93e26359aefad7fb6ee4851958d828c16b,0,51326343192525988136143382979855785783476474407776534721033463010659950298104,18180000,60600000,6060000
7,83558385,0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0,398,2026-02-28 00:00:01,0x2bafc195e509e9988ca946fb180c0b5f4f101b396b459d6a4e1961a5a6824268,0x223fb39cdb365b469523aff98ccef4e3bc072050,0x45730f93e26359aefad7fb6ee4851958d828c16b,0,51326343192525988136143382979855785783476474407776534721033463010659950298104,12999000,43330000,4333000
8,83558385,0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0,408,2026-02-28 00:00:01,0xc5f8a004f3e92585aa116671cff00f4c2d1ddf05cd2ea6876ced4750c709c3d1,0xb29501b374eeab2a080680285bd8da4a9fc7d1e7,0x45730f93e26359aefad7fb6ee4851958d828c16b,0,51326343192525988136143382979855785783476474407776534721033463010659950298104,4578000,15260000,1526000
9,83558385,0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0,418,2026-02-28 00:00:01,0xc4abfa788b3d1a667e155c027e62bdee12609b03f378ef7c132bd85f51468eb3,0xd37ab1a5eeb25696c568e20e94af72b8d3422358,0x45730f93e26359aefad7fb6ee4851958d828c16b,0,51326343192525988136143382979855785783476474407776534721033463010659950298104,4019154,13859151,1385915


In [100]:
token_lookup_dict['16581159092118702962412590502551978746865251171601054389828093232030979416235']

('0x09cbe3e796661a1d820580145488ad2ccb9ad1e720efcd64b448bb77b97007c1',
 'us-strikes-iran-by-february-27-2026',
 ['Politics',
  'Iran',
  'Middle East',
  'Geopolitics',
  'World',
  'Parent For Derivative',
  'HFC'],
 '2026-01-31T00:00:00Z')

In [8]:
trades_extracted_df = pd.read_parquet('../data/trades_polygon/trades_extracted.parquet')
print(len(trades_extracted_df))
pd.concat([trades_extracted_df.head(3), trades_extracted_df.tail(3)])

219147


,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,side,price,quantity,usdc_amount,question,slug,buyer,seller,aggressor,tags
0,0x29a571f32b75e44d7f72cb7d2f49ae02abebfd97b9df...,741,83601589,1772323209,2026-03-01,0xc642ecc9516cb80f78d1bf54a61c79505740b6093e13...,9158448397456600854896064495796460075600774357...,Down,SELL,0.720,50.00000,36.000000,S&P 500 (SPX) Opens Up or Down on March 2?,spx-opens-up-or-down-on-march-2-2026,0x3bbd2534ff00d935a7949a0d45610e2ad270508d,0x2e66236984af3e7f4b011f1318b253cb3b03e1b2,0x2e66236984af3e7f4b011f1318b253cb3b03e1b2,"[Finance, Up or Down, Hide From New, Daily, In..."
1,0xed742fff57afdb1e1b88dbeaa76aba1ce64a55590bcc...,2801,83601594,1772323219,2026-03-01,0x1dbfa21016a7ab2ddff508ca194550e9b8c0833e90c4...,2607855515135026274569874606710697482374428770...,Thunder,SELL,0.900,11.05000,9.945000,Thunder vs. Mavericks,nba-okc-dal-2026-03-01,0x77d32628ba47e23666c767c2f29d30dde341f4d6,0xda821dc3615bfb6af76cac365322cd83d16d8524,0xda821dc3615bfb6af76cac365322cd83d16d8524,"[Sports, Basketball, NBA, Games]"
2,0x628999da3498c7a563a96522f347a2239eb453f8cf81...,2633,83601599,1772323229,2026-03-01,0x9364314112ffeef45391ae3c5ab9262943a75fbdf616...,4127628206817807002548729195164589436100957046...,76ers,SELL,0.250,3.00000,0.750000,76ers vs. Celtics,nba-phi-bos-2026-03-01,0x6ac5bb06a9eb05641fd5e82640268b92f3ab4b6e,0xada56a3f1a9d7d3fe7ec8cbcaa0865109c57db70,0xada56a3f1a9d7d3fe7ec8cbcaa0865109c57db70,"[Sports, Basketball, NBA, Games]"
219144,0xad4f57c9bf3a4c4f84d377546a0a85aa4a406c8cdfbe...,1226,83687977,1772495985,2026-03-02,0x1a4eb7498b6994f981ad5b3076fbca0a359254b3968a...,5283327991947492052861827485084967659167121894...,Norfolk State Spartans,SELL,0.100,5.55555,0.555555,Norfolk State Spartans vs. Morgan State Bears,cbb-norfst-morgst-2026-03-02,0xe70ab7d438d9b9e45687134a9b3429881c3c6de8,0xa51540d5366216fdda233cb20510bd907f356202,0xa51540d5366216fdda233cb20510bd907f356202,"[Sports, Basketball, NCAA, Games, NCAA Basketb..."
219145,0xb6afbed3dc3d0fdf7f932d77acbe91b0cf8b9b1bf3a9...,1370,83687977,1772495985,2026-03-02,0x293a3a3a007a9a524868278a1a6b4af7dfec7a560970...,9357817481832696010338500214978662783368206264...,Over,SELL,0.003,30.09000,0.090270,Estudiantes de La Plata vs. CA Vélez Sarsfield...,arg-est-vel-2026-03-02-total-2pt5,0x2005d16a84ceefa912d4e380cd32e7ff827875ea,0xed7ef2281dc864272ceaa4fe703184b73183c090,0xed7ef2281dc864272ceaa4fe703184b73183c090,"[Sports, Soccer, Games, Argentina Primera Divi..."
219146,0x1099fb4b3fb10c22ae7d7672634512a34e220acecdb6...,1746,83687983,1772495997,2026-03-02,0xb529f668ece4bca3d05ed34864e04d30f41061a17795...,1060608612934668454474938418563989406526963148...,Red Wings,SELL,0.999,29.41000,29.380590,Red Wings vs. Predators,nhl-det-nsh-2026-03-02,0xf7f0b0b1e9c0fe02ccad926916ee31aef74b912c,0x1024b1e479fa7b3da8aaf5670d3fe87a993f9651,0x1024b1e479fa7b3da8aaf5670d3fe87a993f9651,"[Sports, NHL, Hockey, Games]"


In [10]:
# trades_extracted_df[trades_extracted_df['tx_hash'] == '0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0']
trades_extracted_df.groupby('aggressor')['aggressor'].value_counts().head(4)

aggressor
0x00000db3f2fb2086866753b6e41e426a5312e225     1
0x000269109ef14628e4a4a406acc548487ff1103e    10
0x00030670d970058b29b1ab4f045e8db0471ac15c     3
0x00061707080cc568f5ab76304b82c9dab2402caf     1
Name: count, dtype: int64

In [19]:
pd.options.display.max_colwidth = None
#trades_extracted_df[trades_extracted_df['aggressor'] == '0x00000db3f2fb2086866753b6e41e426a5312e225']
trades_extracted_df[trades_extracted_df['tx_hash'] == '0x52119042f21c957754fb74a1710fc6756ad18d154e217e8557c33f70cda353d6']

,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,side,price,quantity,usdc_amount,question,slug,buyer,seller,aggressor,tags
79731,0x52119042f21c957754fb74a1710fc6756ad18d154e217e8557c33f70cda353d6,2997,83650202,1772420435,2026-03-02,0x8f636e748ba8beaab993e371a80d3b99cdf3da6d14dec3d7be823463a2ef03ca,10478019595103930357606511442146391867751885681696977370999614290245877047457,Over,SELL,0.99,50.0,49.5,Wichita State Shockers vs. UTSA Roadrunners: O/U 151.5,cbb-wichst-utsa-2026-03-01-total-151pt5,0xea22ea2400fb8a0b71a47f4f52a29c56b63f747a,0x00000db3f2fb2086866753b6e41e426a5312e225,0x00000db3f2fb2086866753b6e41e426a5312e225,"[Sports, Basketball, NCAA, Games, NCAA Basketball]"


In [22]:
trades_raw[trades_raw['tx_hash'] == '0x52119042f21c957754fb74a1710fc6756ad18d154e217e8557c33f70cda353d6']

,block_number,tx_hash,log_index,block_timestamp,order_hash,maker,taker,maker_asset_id,taker_asset_id,maker_amount_filled,taker_amount_filled,fee
1090711,83650202,0x52119042f21c957754fb74a1710fc6756ad18d154e217e8557c33f70cda353d6,2997,1772420435,0xb42050a7c8e1bf91df15503f089a4eb8ca4301a7cd1133299e3609dd5a263a59,0xea22ea2400fb8a0b71a47f4f52a29c56b63f747a,0x00000db3f2fb2086866753b6e41e426a5312e225,0,10478019595103930357606511442146391867751885681696977370999614290245877047457,49500000,50000000,50505
1090712,83650202,0x52119042f21c957754fb74a1710fc6756ad18d154e217e8557c33f70cda353d6,3001,1772420435,0x08430f24f43aaff99ff73ad2b29dd4cf16ae683966282b54afac77e4b6332caf,0x00000db3f2fb2086866753b6e41e426a5312e225,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,0,12519780143985910467736763519100067994653209874863801928282938522146345263041,500000,50000000,5000000


In [26]:
import datetime as dt
dt.datetime.fromtimestamp(1772420435)

datetime.datetime(2026, 3, 2, 4, 0, 35)

In [22]:
# get counts grouped by (maker_asset_id, taker_asset_id) 
pair_counts = (
    trades_raw
    .groupby(["maker_asset_id", "taker_asset_id"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(len(pair_counts))

pair_counts.head(20)

33167


,maker_asset_id,taker_asset_id,count
8288,0,37975265083682450969967223199653164268542375291978582835346444673615244164455,39882
8547,0,39317885422026394259056328144566743331998444273202427934141325790266108570112,38930
9970,0,46746822541461330721074821991383617225657173789584499857683753079229786702095,20861
14640,0,70771354585365381988139008309072205730081182435161568795508496003376222185889,18727
25736,39317885422026394259056328144566743331998444273202427934141325790266108570112,0,18423
25579,37975265083682450969967223199653164268542375291978582835346444673615244164455,0,17956
1624,0,10832696757358093775468120009000761778513405247768868107262967513475277652998,14289
26649,46746822541461330721074821991383617225657173789584499857683753079229786702095,0,13729
16533,0,80733527800710944363886141911868517748137807225998893261323072005352734555244,13011
29618,70771354585365381988139008309072205730081182435161568795508496003376222185889,0,12210


In [61]:
pc = pair_counts.copy()

# token is maker_asset_id unless it is 0, then taker_asset_id
pc["asset_token_id"] = (
    pc["maker_asset_id"].astype("string")
    .where(pc["maker_asset_id"].astype("string") != "0", pc["taker_asset_id"].astype("string"))
)

pair_counts_enriched = pc.join(token_lookup, on="asset_token_id", how="left")
pair_counts_enriched.head(1)

,maker_asset_id,taker_asset_id,count,asset_token_id,condition_id,market_slug,tags,end_date_iso
8288,0,37975265083682450969967223199653164268542375291978582835346444673615244164455,39882,37975265083682450969967223199653164268542375291978582835346444673615244164455,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,khamenei-out-as-supreme-leader-of-iran-by-february-28,"[Politics, Iran, Middle East, Geopolitics, World]",2026-02-28T00:00:00Z


In [77]:
tags_exploded = (
    pair_counts_enriched.assign(
        tags=lambda d: d["tags"].map(lambda x: x if isinstance(x, list) else [])
    )
    .explode("tags", ignore_index=True)
    .dropna(subset=["tags"])
)

counts_by_tag = (
    tags_exploded.groupby("tags", dropna=False)["count"]
    .sum()
    .sort_values(ascending=False)
    .reset_index(name="count")
)

print(counts_by_tag)

                              tags    count
0                    Hide From New  7831720
1                           Crypto  7830890
2                        Recurring  7830796
3                    Crypto Prices  7830716
4                       Up or Down  7762604
5                          Bitcoin  5381796
6                               5M  5258674
7                              15M  1982556
8                         Ethereum  1289251
9                           Solana   636230
10                          Sports   608071
11                           Games   607944
12                          Ripple   523507
13                             XRP   523507
14                              1H   487013
15                      Basketball   266583
16                        Politics   182021
17                     Geopolitics   177776
18                     Middle East   167902
19                            Iran   157251
20                          Soccer   148561
21                             N

In [3]:
counts = trades_raw["order_hash"].value_counts(dropna=True)
dup_hashes = counts[counts >= 2]

if dup_hashes.empty:
    print("No duplicated order_hash found.")
else:
    h = dup_hashes.index[0]
    two_trades = trades_raw.loc[trades_raw["order_hash"] == h].head(2)

    cols = [c for c in [
        "order_hash", "tx_hash", "block_number", "block_timestamp",
        "side", "price", "quantity", "buyer", "seller"
    ] if c in two_trades.columns]

    print(f"Found duplicated order_hash: {h} (count={int(dup_hashes.iloc[0])})")
    display(two_trades[cols])

Found duplicated order_hash: 0xc6c1ca55163f0de1e3d1f82a4574cd74d1842537ca62f630131c6726deabe239 (count=1796)


,order_hash,tx_hash,block_number,block_timestamp
447552,0xc6c1ca55163f0de1e3d1f82a4574cd74d1842537ca62...,0x6b80380f5f60536d5658d27e749c942e946266bbd5b1...,82913141,1770946305
447591,0xc6c1ca55163f0de1e3d1f82a4574cd74d1842537ca62...,0xb0030a69b30d7f8cfccf93ed0a8eac36f9f2d5b6e496...,82913141,1770946305


In [4]:
dup_orders = trades_raw[trades_raw['order_hash'] == '0xc6c1ca55163f0de1e3d1f82a4574cd74d1842537ca62f630131c6726deabe239']

In [13]:
pd.to_datetime(1770946305, unit='s')


Timestamp('2026-02-13 01:31:45')

In [17]:
# colwidth = None
pd.options.display.max_colwidth = None
trades_raw.head(10)

,block_number,tx_hash,log_index,block_timestamp,order_hash,maker,taker,maker_asset_id,taker_asset_id,maker_amount_filled,taker_amount_filled,fee
0,83558385,0x80edc41388e388c84328fcd5ece2d1b069f6cc85e43d67fde2e4d848a79e9f24,328,1772236801,0xe2b0a7edd5c6c259d29b673ea229da5634d191a40913c298b21c504612a37cc6,0xab15040749c9a4ead610d6f0ba8f8da669137eac,0x076ca0207aaa48365da2eb6a917edcbdee5e4e7b,0,16581159092118702962412590502551978746865251171601054389828093232030979416235,9580040,258920000,0
1,83558385,0x80edc41388e388c84328fcd5ece2d1b069f6cc85e43d67fde2e4d848a79e9f24,330,1772236801,0x1ab32e2e558ac541e888533f92cf16f3e2fa99aa152404209b3b39058d39ccfc,0x076ca0207aaa48365da2eb6a917edcbdee5e4e7b,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,16581159092118702962412590502551978746865251171601054389828093232030979416235,0,258920000,9580040,0
2,83558385,0x47950bf4a4dc83fb7b82cf76c91aa68e275e92ce5b7e29c412710aa119a71f1d,344,1772236801,0x7a04a9bc97e7a5b75d12c41cd2b443eb914bdc03b84067559d28b4e867cd5f60,0x111f73e91f85b6fe4de1ddec3de2fe32122e355b,0xf6963d4cdbb6f26d753bda303e9513132afb1b7d,0,8629673814310781337262814398651414822607314895449171908987170554118708719632,7255600,10670000,502117
3,83558385,0x47950bf4a4dc83fb7b82cf76c91aa68e275e92ce5b7e29c412710aa119a71f1d,354,1772236801,0xcd829b6b817e9301138bc4738c1c7f7378f1f1e1ebbfe3d14f0fe3ea75e49bd7,0x36a708485c603a477804ec74122475647629f03b,0xf6963d4cdbb6f26d753bda303e9513132afb1b7d,0,8629673814310781337262814398651414822607314895449171908987170554118708719632,6400000,10000000,562500
4,83558385,0x47950bf4a4dc83fb7b82cf76c91aa68e275e92ce5b7e29c412710aa119a71f1d,358,1772236801,0x04aeb6a416faeb43d09276402977037869d9e69081f770bc24b53c6a60d61630,0xf6963d4cdbb6f26d753bda303e9513132afb1b7d,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,0,58910049878019002824701532570780342958758796534587879434389379308514509314738,7014400,20670000,2067000
5,83558385,0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0,378,1772236801,0xefe3172a3f9f9b6edf70c7e4bae007217be021e7ab97f79fc449c7496e95e867,0xefde78f38169634477f6ff46572128b6d21a6505,0x45730f93e26359aefad7fb6ee4851958d828c16b,0,51326343192525988136143382979855785783476474407776534721033463010659950298104,2883000,9610000,961000
6,83558385,0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0,388,1772236801,0x842b8ed0d2493b405f1eba864529c49a936304be9bb9e8e36fb986e523f89f26,0xd37ab1a5eeb25696c568e20e94af72b8d3422358,0x45730f93e26359aefad7fb6ee4851958d828c16b,0,51326343192525988136143382979855785783476474407776534721033463010659950298104,18180000,60600000,6060000
7,83558385,0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0,398,1772236801,0x2bafc195e509e9988ca946fb180c0b5f4f101b396b459d6a4e1961a5a6824268,0x223fb39cdb365b469523aff98ccef4e3bc072050,0x45730f93e26359aefad7fb6ee4851958d828c16b,0,51326343192525988136143382979855785783476474407776534721033463010659950298104,12999000,43330000,4333000
8,83558385,0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0,408,1772236801,0xc5f8a004f3e92585aa116671cff00f4c2d1ddf05cd2ea6876ced4750c709c3d1,0xb29501b374eeab2a080680285bd8da4a9fc7d1e7,0x45730f93e26359aefad7fb6ee4851958d828c16b,0,51326343192525988136143382979855785783476474407776534721033463010659950298104,4578000,15260000,1526000
9,83558385,0x48e6a3e0c3bb9ca8f49b221ad25a2a94367e555551278d4611d4a58569e28fa0,418,1772236801,0xc4abfa788b3d1a667e155c027e62bdee12609b03f378ef7c132bd85f51468eb3,0xd37ab1a5eeb25696c568e20e94af72b8d3422358,0x45730f93e26359aefad7fb6ee4851958d828c16b,0,51326343192525988136143382979855785783476474407776534721033463010659950298104,4019154,13859151,1385915


In [ ]:
found_trades = trades_raw[trades_raw['tx_hash'].isin(HASHES)]
len(found_trades)

In [ ]:
# list hashes that were not found
not_found_hashes = [h for h in HASHES if h not in found_trades['tx_hash'].values]
print(len(not_found_hashes))
not_found_hashes

In [ ]:
trade_df = pd.read_parquet('/Users/vobornij/projects/polymarket/data/trades_polygon/trades_extracted.parquet')
print(f"Number of trades: {len(trade_df)}")

trade_df['block_timestamp'] = pd.to_datetime(trade_df['block_timestamp'], unit='s')
trade_df.head(1)

In [24]:
markets = pd.read_parquet('/Users/vobornij/projects/polymarket/data/markets/2026-02.parquet')
print(len(markets))
markets.head(1)

117605


,end_date_iso,condition_id,market_json
0,2026-02-01T00:00:00Z,0x00005c113846891686d99ff1d145ded8bdccfa028df0fdf15d66f11d176c49b1,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2026-01-12T23:07:49Z"",""minimum_order_size"":5,""minimum_tick_size"":0.001,""condition_id"":""0x00005c113846891686d99ff1d145ded8bdccfa028df0fdf15d66f11d176c49b1"",""question_id"":""0x8f2bd95c903c8b19bc322d0cf2780bc7aedd5ce7a2cd161637f466b6858286eb"",""question"":""Will Jonathan Micallef win by KO or TKO?"",""description"":""This market will resolve to \""Yes\"" if Jonathan Micallef defeats Oban Elliott at UFC 325: Volkanovski vs. Lopes 2, scheduled for January 31, 2026, by KO or TKO, including referee stoppage, doctor stoppage, or corner stoppage. Otherwise, it will resolve to \""No.\""\n\nIf the fight ends in a draw or by disqualification (either fighter), it will resolve \""No.\""\n\nIf the bout is ruled a No Contest, not scored, canceled, or postponed beyond February 14, 2026, this market will resolve \""50-50.\""\n\nThe resolution source for this market will be official information from the UFC."",""market_slug"":""ufc-oba-jon21-2026-01-31-micallef-win-by-ko-tko"",""end_date_iso"":""2026-02-01T00:00:00Z"",""game_start_time"":""2026-02-01T00:00:00Z"",""seconds_delay"":3,""fpmm"":"""",""maker_base_fee"":0,""taker_base_fee"":0,""notifications_enabled"":true,""neg_risk"":false,""neg_risk_market_id"":"""",""neg_risk_request_id"":"""",""icon"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/ufc-logo-37bbbd28e6.png"",""image"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/ufc-logo-37bbbd28e6.png"",""rewards"":{""rates"":null,""min_size"":0,""max_spread"":0},""is_50_50_outcome"":false,""tokens"":[{""token_id"":""39193516886599060075501550109703179474527630775322669834266419407202332831463"",""outcome"":""Yes"",""price"":0,""winner"":false},{""token_id"":""24644942608518363885242446606120538480406676251324592596786750145524562656428"",""outcome"":""No"",""price"":1,""winner"":true}],""tags"":[""Sports"",""UFC"",""Games"",""UFC 325""]}"


In [39]:
MARKET = '0xff3f71b8ed2b233cd6581d61b44edc5dd5d094830445fb6f89dc61290573cade'
trades_market = trade_df[trade_df['condition_id'] == MARKET]
print(f"Number of trades for market {MARKET}: {len(trades_market)}")
trades_market.head(1)


NameError: name 'trade_df' is not defined

In [ ]:
TX_HASH = '0x98b4cc92db6284cc907e0a83af4d65d4d33a247c7bc684281cd43ac16bc51314'
matching_trades = trade_df[trade_df['tx_hash'] == TX_HASH]
print(len(matching_trades))
matching_trades

In [ ]:
trades_market[['tx_hash', 'block_timestamp', 'side', 'price', 'quantity', 'buyer', 'seller']]